# Hotels-50K — Feasibility, round 2

The first probe measured **aggregate** liveness (75.7% on the travel CDN) and called STOP.
That threshold was wrong: aggregate liveness is not the quantity that decides this project.

What actually matters is **per-hotel survival**. A hotel is only useless if its gallery drops
below roughly 5 images. Under independent per-image loss at 75.7%, simulation says the median
hotel keeps ~18 images and essentially none fall under 8 — the project is fine.

The danger is if 404s **cluster by hotel** (a whole property purged from the CDN). Then the
same 24% loss wipes out entire hotels instead of thinning every gallery evenly.

This notebook answers three questions:
1. Is the loss independent per image, or clustered per hotel?
2. What resolution are the images actually? (mean 19 KB is suspiciously small)
3. Is a higher-resolution URL variant available on the same CDN?

## 1. Setup — reload metadata and rebuild the candidate selection

In [ ]:
!pip install -q pandas requests pillow

import os, subprocess, tarfile, random, time, json, io
import concurrent.futures as cf
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from PIL import Image

REPO = "Hotels-50K"
if Path.cwd().name == REPO:
    os.chdir("..")
if not Path(REPO).exists():
    subprocess.run(["git", "clone", "--depth", "1", "-q",
                    f"https://github.com/GWUvision/{REPO}.git"], check=True)
os.chdir(REPO)
if not Path("input/dataset").exists():
    with tarfile.open("input/dataset.tar.gz") as t:
        t.extractall("input/")

COLS = ["image_id", "hotel_id", "image_url", "image_source", "upload_timestamp"]
DATA = Path("input/dataset")
train  = pd.read_csv(DATA / "train_set.csv", names=COLS, header=None, dtype=str)
test   = pd.read_csv(DATA / "test_set.csv", dtype=str)
hotels = pd.read_csv(DATA / "hotel_info.csv", dtype=str)

HEADERS = {"User-Agent": "Mozilla/5.0 (research; hotel-recognition coursework)"}

MIN_TRAIN, N_HOTELS, CAP = 15, 1000, 40
per  = train.groupby("hotel_id").size()
cand = [h for h in test.hotel_id.unique() if per.get(h, 0) >= MIN_TRAIN]
sel  = set(cand[:N_HOTELS])
gallery = train[train.hotel_id.isin(sel)].groupby("hotel_id").head(CAP)

print(f"candidate hotels {len(sel):,} | gallery {len(gallery):,} images")

## 2. The decisive test — is loss independent or clustered?

Pick 40 hotels at random and probe **every** gallery URL they own. Then compare the observed
spread of per-hotel survival against the binomial spread you'd expect under independence.

- Observed variance ≈ binomial variance → loss is random. **Project is fine.**
- Observed variance >> binomial → loss clusters by hotel. **Some hotels are dead.**

In [ ]:
N_HOTELS_PROBE = 40
WORKERS = 24
SEED = 0

def alive(url, timeout=12):
    try:
        r = requests.head(url, timeout=timeout, headers=HEADERS, allow_redirects=True)
        if r.status_code == 405:
            r = requests.get(url, timeout=timeout, headers=HEADERS,
                             stream=True, allow_redirects=True)
            r.close()
        return bool(r.ok)
    except Exception:
        return False

rng = random.Random(SEED)
probe_hotels = rng.sample(sorted(sel), N_HOTELS_PROBE)
sub = gallery[gallery.hotel_id.isin(probe_hotels)].copy()
print(f"probing {len(sub):,} URLs across {N_HOTELS_PROBE} hotels...")

t0 = time.time()
with cf.ThreadPoolExecutor(WORKERS) as ex:
    sub["alive"] = list(ex.map(alive, sub.image_url))
print(f"done in {time.time()-t0:.0f}s\n")

g = sub.groupby("hotel_id").agg(n=("alive", "size"), k=("alive", "sum"))
g["surv"] = g.k / g.n
p_hat = g.k.sum() / g.n.sum()

print(f"pooled survival p = {p_hat:.3f}")
print(f"per-hotel survival: median {g.surv.median():.2f}  "
      f"p10 {g.surv.quantile(.1):.2f}  min {g.surv.min():.2f}")
print(f"hotels with 0 surviving images  : {(g.k == 0).sum()} / {len(g)}")
print(f"hotels with < 5 surviving images: {(g.k < 5).sum()} / {len(g)}")
print(f"hotels with < 8 surviving images: {(g.k < 8).sum()} / {len(g)}")

In [ ]:
# --- overdispersion test: observed vs binomial variance ---
exp_var = (p_hat * (1 - p_hat) / g.n).mean()     # mean binomial variance of the rate
obs_var = g.surv.var(ddof=1)
ratio   = obs_var / exp_var

# chi-square dispersion statistic
chi2 = (((g.k - g.n * p_hat) ** 2) / (g.n * p_hat * (1 - p_hat))).sum()
dof  = len(g) - 1

print(f"observed variance of per-hotel survival : {obs_var:.5f}")
print(f"binomial expectation under independence : {exp_var:.5f}")
print(f"overdispersion ratio                    : {ratio:.2f}x")
print(f"chi2 = {chi2:.1f} on {dof} dof  (ratio {chi2/dof:.2f})")
print()
if ratio < 2:
    print(">> LOSS IS ESSENTIALLY RANDOM. Every gallery thins evenly.")
    print(">> Compensate by raising CAP; the project stands.")
elif ratio < 5:
    print(">> MILD CLUSTERING. Filter hotels by surviving count after download.")
else:
    print(">> STRONG CLUSTERING. Whole hotels are gone from the CDN.")
    print(">> Over-select hotels heavily, or switch to Kaggle Hotel-ID.")

In [ ]:
# --- what the sample implies for the full selection ---
S = 4000
rs = np.random.default_rng(1)

mix = gallery.groupby(["hotel_id", "image_source"]).size().unstack(fill_value=0)
for col in ("travel_website", "traffickcam"):
    if col not in mix:
        mix[col] = 0

# use the measured pooled rate for travel_website; traffickcam probed at 100%
p_tw = p_hat
sim = (rs.binomial(mix.travel_website.values[None, :], p_tw, size=(S, len(mix)))
       + mix.traffickcam.values[None, :])

print(f"projected surviving gallery: {sim.mean():.0f} imgs/hotel "
      f"({sim.sum(1).mean():,.0f} total, {100*sim.sum(1).mean()/len(gallery):.0f}% of manifest)")
for k in (3, 5, 8, 10):
    print(f"  hotels expected below {k:2d} images: {(sim < k).mean(0).sum():.1f} of {len(mix)}")

## 3. Resolution audit

Mean sizes from round 1: travel_website **19 KB**, traffickcam **262 KB**. A 14x gap.
If the gallery is thumbnails and the queries are full photos, that is a resolution mismatch
on a task that depends on fine detail — bedspread patterns, artwork, fixtures.

This matters more than the 404 rate. Measure actual pixels.

In [ ]:
def dims(url, timeout=15):
    try:
        r = requests.get(url, timeout=timeout, headers=HEADERS)
        if not r.ok:
            return None
        im = Image.open(io.BytesIO(r.content))
        return im.size + (len(r.content) / 1024,)
    except Exception:
        return None

rng2 = random.Random(7)
out = {}
for src in ("travel_website", "traffickcam"):
    pool = train[train.image_source == src].image_url.tolist()
    urls = rng2.sample(pool, 40)
    with cf.ThreadPoolExecutor(16) as ex:
        res = [r for r in ex.map(dims, urls) if r]
    if not res:
        print(f"[{src}] no successful fetches"); continue
    w, h, kb = zip(*res)
    out[src] = res
    print(f"[{src}]  n={len(res)}")
    print(f"   width  : median {np.median(w):.0f}  range {min(w)}-{max(w)}")
    print(f"   height : median {np.median(h):.0f}  range {min(h)}-{max(h)}")
    print(f"   size   : median {np.median(kb):.0f} KB")
    print(f"   short side median: {np.median([min(a,b) for a,b in zip(w,h)]):.0f} px")
    print()

print(">> Short side >= 224 px means you can feed a standard backbone without upsampling.")
print(">> Below that, the gallery is genuinely lower-information than the queries.")

## 4. Is a bigger variant available?

The travel CDN uses a size suffix before `.jpg` (the sampled URLs end in `_b.jpg`).
Other letters may serve larger renditions of the same photo. Test empirically — do not assume.

In [ ]:
import re

SUFFIXES = ["_b", "_y", "_z", "_t", "_s", "_l", "_w", ""]

base_urls = (train[train.image_source == "travel_website"]
             .image_url.sample(12, random_state=3).tolist())

def variant(url, suf):
    return re.sub(r"_[a-z]\.jpg$", f"{suf}.jpg", url) if suf else re.sub(r"_[a-z]\.jpg$", ".jpg", url)

rows = []
for suf in SUFFIXES:
    ok, ws, kbs = 0, [], []
    for u in base_urls:
        r = dims(variant(u, suf))
        if r:
            ok += 1; ws.append(max(r[0], r[1])); kbs.append(r[2])
    rows.append(dict(suffix=suf or "(none)", alive=f"{ok}/{len(base_urls)}",
                     median_long_side=int(np.median(ws)) if ws else 0,
                     median_kb=round(float(np.median(kbs)), 1) if kbs else 0.0))

df = pd.DataFrame(rows).sort_values("median_long_side", ascending=False)
print(df.to_string(index=False))
print()
print(">> If a suffix serves a materially larger image at a similar hit rate,")
print(">> rewrite the manifest URLs to use it before downloading.")

## 5. Revised verdict

Replace the round-1 rule. Aggregate liveness was never the right gate.

In [ ]:
dead_hotels   = int((g.k < 5).sum())
dead_frac     = dead_hotels / len(g)
proj_per_hotel = float(sim.mean())

print("=" * 64)
print(f"pooled survival        : {p_hat:.1%}")
print(f"overdispersion         : {ratio:.2f}x binomial")
print(f"hotels under 5 images  : {dead_hotels}/{len(g)}  ({dead_frac:.1%})")
print(f"projected imgs/hotel   : {proj_per_hotel:.1f}")
print("-" * 64)

if dead_frac > 0.25 or proj_per_hotel < 6:
    print("STOP — switch to Kaggle Hotel-ID (hotel-id-to-combat-human-trafficking-2022-fgvc9)")
elif dead_frac > 0.10 or ratio > 5:
    print("PROCEED WITH OVER-SELECTION")
    print(f"  select ~{int(N_HOTELS / (1 - dead_frac) * 1.1)} hotels to land {N_HOTELS} usable ones,")
    print("  then drop any hotel with fewer than 5 downloaded images before indexing.")
else:
    print("GO — loss is diffuse, not concentrated.")
    print(f"  raise CAP from {CAP} to {int(CAP / p_hat) + 1} to hold gallery size constant,")
    print("  and drop hotels under 5 surviving images post-download (expected to be few).")
print("=" * 64)

## 6. What to write in the proposal either way

The 404 rate is not a footnote — it is a finding, and it is defensible material for the
report's limitations section:

- The dataset is distributed as **URLs, not images**, so it decays over time. Report the
  survival rate you measured and the date you measured it. That makes your work reproducible
  in a way the original paper is not.
- If you carry on with Hotels-50K, state the **effective** dataset size after decay, not the
  nominal 1.1M.
- The gallery/query resolution gap measured in section 3 is a genuine domain shift between
  professional listing photos and phone captures. If it is large, it is worth an experiment
  rather than an apology: downsample the queries to gallery resolution and see how much of
  the retrieval gap it explains.
- Fallback stays available: Kaggle Hotel-ID ships actual image files, so it cannot rot. You
  would lose the occlusion sweep and the travel-website/TraffickCam domain-shift experiment,
  and keep everything else.